# Joint Input Robustness of Budget-Constrained Tool Calling
## AI465 Agentic AI Research Project
**Primary Academic Reference:** *On the Robustness of Agentic Function Calling* — Ella Rabinovich & Ateret Anaby-Tavor (TrustNLP 2025)

This notebook demonstrates and evaluates the **Joint Input Robustness** of an LLM agent under simultaneous **query paraphrasing** and **toolkit scaling**, comparing baseline full toolkits against **Sentence Transformer-based semantic tool shortlisting** (`all-MiniLM-L6-v2`).


## 2. Research Question and Hypotheses

### Primary Research Question
> **How does the combination of paraphrased user requests and an expanded set of available tools affect an LLM's tool-calling accuracy and token cost? Can semantic tool shortlisting reduce token consumption while preserving accuracy?**

### Hypotheses
- **Hypothesis 1 (Perturbation Sensitivity)**: Linguistic paraphrasing of user queries (Condition B) degrades tool-selection fidelity compared to canonical phrasing (Condition A) in the presence of distractor tools.
- **Hypothesis 2 (Linear Token Inflation)**: Presenting an expanded 20-tool library inflates input prompt tokens by >600% compared to a pruned 3-tool candidate set.
- **Hypothesis 3 (Shortlist Efficiency & Recovery)**: Deterministic semantic shortlisting (Condition C) reduces prompt token consumption by 70–85% relative to Condition B while preserving tool selection accuracy, provided shortlist recall is sufficiently high.

## 3. Academic Background and References

This investigation builds upon and critically evaluates three core literature foundations:

1. **Ella Rabinovich and Ateret Anaby-Tavor (2025)**  
   *"On the Robustness of Agentic Function Calling"*, Proceedings of the 5th Workshop on Trustworthy NLP (TrustNLP 2025), pages 298–304. [ACL Anthology](https://aclanthology.org/2025.trustnlp-main.20/)  
   *Core Finding*: LLM tool selection degrades under minor prompt phrasing perturbations due to surface-form overfitting to tool docstrings.
2. **Shishir G. Patil et al. (2025)**  
   *"The Berkeley Function Calling Leaderboard"*, PMLR v267: patil25a. [BFCL Paper](https://proceedings.mlr.press/v267/patil25a.html)  
   *Core Finding*: Standardized single-turn and AST argument evaluation metrics; demonstrated accuracy decline when appending irrelevant distractor functions.
3. **ToolRobustBench (2026)**  
   *"Evaluating the Multi-Dimensional Robustness of Tool-Augmented LLMs"*, [arXiv:2608.23635](https://arxiv.org/abs/2608.23635)  
   *Core Finding*: Argues that agent robustness benchmarks must account for operational token costs alongside raw accuracy.

## 4. Environment and Configuration
Import core libraries, initialize path references, and load configuration parameters.

In [1]:
import sys
import os
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from src.config import (
    GEMINI_MODEL,
    MAX_API_BUDGET,
    MAX_TOKEN_BUDGET,
    RANDOM_SEED,
    is_api_configured,
    get_masked_api_key
)

print(f"Project Root:       {PROJECT_ROOT}")
print(f"Target Model:       {GEMINI_MODEL}")
print(f"Max Call Budget:    {MAX_API_BUDGET} requests")
print(f"Max Token Budget:   {MAX_TOKEN_BUDGET} tokens")
print(f"Random Seed:        {RANDOM_SEED}")

Project Root:       C:\Users\ashis\Music\Desktop\AA project\Joint_Input_Robustness
Target Model:       gemini-3.5-flash
Max Call Budget:    150 requests
Max Token Budget:   500000 tokens
Random Seed:        42


## 5. API-Key Validation (Without Printing Secrets)
Verify API key presence safely without logging secrets or exposing credentials.

In [2]:
api_ready = is_api_configured()
masked_key = get_masked_api_key()

print(f"API Key Configured: {api_ready}")
print(f"Masked Key Status:  {masked_key}")

if not api_ready:
    print("\n[!] Notice: No valid Gemini API Key detected. The experiment will execute in dry-run mode.")
else:
    print("\n[+] API Key detected! Real API requests will be logged and cached.")

API Key Configured: True
Masked Key Status:  AQ.A...7AGQ

[+] API Key detected! Real API requests will be logged and cached.


## 6. Dataset Provenance and Limitations

- **Source**: Curated academic benchmark designed for course AI465, modeled on BFCL single-turn function calling standards.
- **License**: Creative Commons Attribution 4.0 International (CC BY 4.0).
- **Scale**: 30 verified tasks across 10 functional domains. For pilot execution under free-tier quotas, a representative 5-task paired subset is evaluated.
- **Transparency**: If API quota limits are reached, the pipeline gracefully checkpoints all completed tasks without synthetic data fabrication.

## 7. Dataset Loading and Schema Validation
Load all candidate tools and evaluation tasks, verifying ground-truth schema conformity.

In [3]:
from src.dataset import load_tasks, load_tools, validate_dataset

tools = load_tools()
tasks = load_tasks()
validation = validate_dataset(tools=tools, tasks=tasks)

print(f"Total Candidate Tools: {len(tools)} (Target: 10, Distractors: 10)")
print(f"Total Benchmark Tasks: {len(tasks)}")
print(f"Schema Validation:     {'PASSED' if validation['valid'] else 'FAILED'}")
if validation['errors']:
    print(f"Errors: {validation['errors']}")

Total Candidate Tools: 24 (Target: 10, Distractors: 10)
Total Benchmark Tasks: 30
Schema Validation:     PASSED


## 8. Tool Definitions: Targets vs Distractors
Inspect representative tools to observe how distractors share overlapping vocabulary.

In [4]:
print("=== SAMPLE TARGET TOOL vs REALISTIC DISTRACTOR ===")
target_tool = next(t for t in tools if t.name == "get_weather")
distractor_tool = next(t for t in tools if t.name == "get_forecast")

print(f"Target Tool:     {target_tool.name}")
print(f"Description:     {target_tool.description}")
print(f"Parameters:      {list(target_tool.parameters.properties.keys())}")
print("-" * 50)
print(f"Distractor Tool: {distractor_tool.name}")
print(f"Description:     {distractor_tool.description}")
print(f"Parameters:      {list(distractor_tool.parameters.properties.keys())}")

=== SAMPLE TARGET TOOL vs REALISTIC DISTRACTOR ===
Target Tool:     get_weather
Description:     Retrieve current real-time atmospheric weather conditions, temperature, and humidity for a specified city.
Parameters:      ['city', 'units']
--------------------------------------------------
Distractor Tool: get_forecast
Description:     Retrieve multi-day future weather projections, meteorological forecasts, and rain probabilities for upcoming days.
Parameters:      ['location', 'days']


## 9. Original vs Paraphrased Queries and Lexical Diversity
Inspect paired queries and measure lexical distance via token-level Jaccard similarity.

In [5]:
from src.paraphrasing import compute_lexical_diversity

lex_metrics = compute_lexical_diversity(tasks)
print(f"Mean Jaccard Similarity:    {lex_metrics['mean_jaccard_similarity']:.4f}")
print(f"Mean Lexical Distance:      {lex_metrics['mean_lexical_distance']:.4f}")
print(f"Mean Character Length Diff: {lex_metrics['mean_character_length_diff']} chars\n")

for t in tasks[:3]:
    print(f"[{t.task_id}] Target: {t.expected_tool_name}")
    print(f"  Orig: '{t.original_query}'")
    print(f"  Para: '{t.paraphrased_query}'\n")

Mean Jaccard Similarity:    0.2705
Mean Lexical Distance:      0.7295
Mean Character Length Diff: 16.43 chars

[task_01] Target: get_weather
  Orig: 'What is the current atmospheric weather in London right now?'
  Para: 'Could you check the current weather conditions and temperature outside in London?'

[task_02] Target: get_weather
  Orig: 'Check the current weather in Tokyo and report in celsius.'
  Para: 'Please provide the prevailing weather conditions in Tokyo with temperature in celsius.'

[task_03] Target: get_forecast
  Orig: 'What is the 5-day weather forecast for Paris?'
  Para: 'Can you retrieve the upcoming meteorological projections for Paris over the next 5 days?'



## 10. Toolkit Expansion and Distractor Architecture
Summarize the 20 tools across functional domains.

In [6]:
df_tools = pd.DataFrame([
    {
        "Name": t.name,
        "Domain": t.domain,
        "Role": "Distractor" if t.is_distractor else "Target",
        "Target Counterpart": t.target_counterpart or "-"
    }
    for t in tools
])
print(df_tools.to_string(index=False))

                       Name     Domain       Role  Target Counterpart
           search_documents  documents     Target                   -
   search_document_by_title  documents Distractor    search_documents
 search_document_by_keyword  documents Distractor    search_documents
            search_products e-commerce     Target                   -
search_products_by_category e-commerce Distractor     search_products
       find_product_by_name e-commerce Distractor     search_products
           search_customers        crm     Target                   -
       get_customer_profile        crm Distractor    search_customers
              search_orders     orders     Target                   -
           get_order_status     orders     Target                   -
          get_order_details     orders Distractor    get_order_status
                get_weather    weather     Target                   -
               get_forecast    weather Distractor         get_weather
         search_rest

## 11. Sentence Transformer Semantic Tool Retrieval (all-MiniLM-L6-v2)
We employ `sentence-transformers/all-MiniLM-L6-v2` to map user queries and standardized tool definitions into a shared 384-dimensional dense semantic embedding space. In Condition C, we retrieve the Top-3 tools via cosine similarity, pruning ~85% of tool schema tokens before invoking the LLM.


In [7]:
from src.tool_shortlisting import SentenceTransformerToolShortlister, TfidfToolShortlister

# Initialize neural semantic shortlister (CPU-friendly, loaded once)
st_shortlister = SentenceTransformerToolShortlister()
test_task = tasks[0]
res = st_shortlister.shortlist(
    query=test_task.paraphrased_query,
    tools=tools,
    expected_tool_name=test_task.expected_tool_name,
    top_k=3
)

print(f"Original Query:     '{test_task.original_query}'")
print(f"Paraphrased Query:  '{test_task.paraphrased_query}'")
print(f"Expected Tool:      '{test_task.expected_tool_name}'")
print(f"Retrieved Top-3:    {res.tool_names}")
print(f"Similarity Scores:  {res.scores}")
print(f"Target in Top-3:    {res.expected_tool_present}")
print(f"Retrieval Latency:  {res.retrieval_latency_ms:.2f} ms")


Original Query:     'What is the current atmospheric weather in London right now?'
Paraphrased Query:  'Could you check the current weather conditions and temperature outside in London?'
Expected Tool:      'get_weather'
Retrieved Top-3:    ['get_weather', 'get_forecast', 'find_restaurants_nearby']
Similarity Scores:  {'get_weather': 0.5378, 'get_forecast': 0.4124, 'find_restaurants_nearby': 0.1693}
Target in Top-3:    True
Retrieval Latency:  127.81 ms



No sentence-transformers model found with name C:\Users\ashis\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2\snapshots\c9745ed1d9f207416be6d2e6f8de32d1f16199bf. Creating a new one with mean pooling.


## 12. Model Client & Function Calling Interface
Initialize `ModelClient` with disk caching and rate-limit backoff.

In [8]:
from src.model_client import ModelClient

client = ModelClient(model_name=GEMINI_MODEL, dry_run=not is_api_configured())
print(f"Model Client initialized for: {client.model_name}")
print(f"Dry Run Active: {client.dry_run}")
print(f"Cache Directory: {PROJECT_ROOT / 'results' / 'cache'}")

Model Client initialized for: gemini-3.5-flash
Dry Run Active: False
Cache Directory: C:\Users\ashis\Music\Desktop\AA project\Joint_Input_Robustness\results\cache


## 13. Small API Smoke Test
Perform a single tool call test to confirm end-to-end connectivity.

In [9]:
smoke_task = tasks[0]
smoke_resp = client.call_tool(
    query=smoke_task.original_query,
    tools=tools
)

print(f"Smoke Test Tool Call:   {smoke_resp.predicted_tool}")
print(f"Predicted Arguments:    {smoke_resp.predicted_arguments}")
print(f"API Success:            {smoke_resp.api_success}")
print(f"Input Tokens:           {smoke_resp.input_token_count} ({smoke_resp.token_count_source})")
print(f"Latency:                {smoke_resp.latency_seconds:.3f}s")

Smoke Test Tool Call:   get_weather
Predicted Arguments:    {'city': 'London'}
API Success:            True
Input Tokens:           2076 (provider_reported)
Latency:                2.088s


## 14. Condition A: Original Queries + Full 20 Tools
Execute or load cached Condition A runs.

In [10]:
from src.experiment import ExperimentRunner

runner = ExperimentRunner(tasks=tasks[:5], tools=tools, client=client)
df_a = runner.run(conditions=["Condition_A"])
print(f"Condition A evaluations completed: {len(df_a[df_a['condition'] == 'Condition_A'])}")

Condition A evaluations completed: 30


## 15. Condition B: Paraphrased Queries + Full 20 Tools
Execute or load cached Condition B runs.

In [11]:
df_b = runner.run(conditions=["Condition_B"])
print(f"Condition B evaluations completed: {len(df_b[df_b['condition'] == 'Condition_B'])}")

Condition B evaluations completed: 30


## 16. Condition C: Paraphrased Queries + Top-3 Shortlist
Execute or load cached Condition C runs.

In [12]:
df_c = runner.run(conditions=["Condition_C"])
print(f"Condition C evaluations completed: {len(df_c[df_c['condition'] == 'Condition_C'])}")

Condition C evaluations completed: 30


## 17. Intermediate Results Saved to Disk
Verify `predictions.csv` and `experiment_results.csv` on disk.

In [13]:
df_all = pd.read_csv(PROJECT_ROOT / "results" / "predictions.csv")
print(f"Total rows in predictions.csv: {len(df_all)}")
print(f"Conditions present: {df_all['condition'].value_counts().to_dict()}")
print(f"Successful calls:   {(df_all['api_success'] == True).sum()}")

Total rows in predictions.csv: 90
Conditions present: {'Condition_A': 30, 'Condition_B': 30, 'Condition_C': 30}
Successful calls:   90


## 18. Evaluation Metrics Computation
Calculate tool selection accuracy, exact match accuracy, and input token consumption across all 30 benchmark tasks.

In [14]:
from src.evaluation import compute_condition_metrics
from src.token_accounting import compute_token_savings

# Compute condition metrics from task-level predictions
metrics_a = compute_condition_metrics(df_all[df_all['condition'] == 'Condition_A'])
metrics_b = compute_condition_metrics(df_all[df_all['condition'] == 'Condition_B'])
metrics_c = compute_condition_metrics(df_all[df_all['condition'] == 'Condition_C'])

savings = compute_token_savings(metrics_b['mean_input_tokens'], metrics_c['mean_input_tokens'])

print('--- 30-TASK BENCHMARK EVALUATION SUMMARY ---')
print(f"Condition A Exact Accuracy: {metrics_a['exact_call_accuracy']*100:.1f}%, Mean Tokens: {metrics_a['mean_input_tokens']:.1f}")
print(f"Condition B Exact Accuracy: {metrics_b['exact_call_accuracy']*100:.1f}%, Mean Tokens: {metrics_b['mean_input_tokens']:.1f}")
print(f"Condition C Exact Accuracy: {metrics_c['exact_call_accuracy']*100:.1f}%, Mean Tokens: {metrics_c['mean_input_tokens']:.1f}")
print(f"Condition C Token Savings:  {savings:.2f}%")


--- 30-TASK BENCHMARK EVALUATION SUMMARY ---
Condition A Exact Accuracy: 93.3%, Mean Tokens: 2080.0
Condition B Exact Accuracy: 90.0%, Mean Tokens: 2082.2
Condition C Exact Accuracy: 96.7%, Mean Tokens: 281.8
Condition C Token Savings:  86.47%


## 19. Paired Statistical Comparisons & Bootstrap Confidence Intervals
Compute paired differences, bootstrap 95% confidence intervals, and McNemar contingency statistics.

In [15]:
from src.evaluation import compute_paired_comparison

p_ab = compute_paired_comparison(df_all, 'Condition_A', 'Condition_B')
p_bc = compute_paired_comparison(df_all, 'Condition_B', 'Condition_C')

print(f"Condition A vs B Paired Diff: {p_ab['mean_difference']:.4f}, 95% CI: {p_ab['bootstrap_ci_95']}, McNemar p: {p_ab['mcnemar_p_value']}")
print(f"Condition B vs C Paired Diff: {p_bc['mean_difference']:.4f}, 95% CI: {p_bc['bootstrap_ci_95']}, McNemar p: {p_bc['mcnemar_p_value']}")


Condition A vs B Paired Diff: 0.0333, 95% CI: [0.0, 0.1], McNemar p: 1.0
Condition B vs C Paired Diff: -0.0667, 95% CI: [-0.1667, 0.0], McNemar p: 0.5


## 20. Visualizations
Generate and display publication figures depicting non-100% distractor degradation and token savings.

In [16]:
from src.visualization import generate_all_figures
from IPython.display import Image, display

generate_all_figures(df_all)

fig_names = [
    'accuracy_comparison.png',
    'token_comparison.png',
    'shortlist_recall.png',
    'task_level_comparison.png',
    'failure_breakdown.png'
]

for name in fig_names:
    p = PROJECT_ROOT / 'figures' / name
    if p.exists():
        print(f"Figure: {name}")
        display(Image(filename=str(p), width=650))


Figure: accuracy_comparison.png
<IPython.core.display.Image object>
Figure: token_comparison.png
<IPython.core.display.Image object>
Figure: shortlist_recall.png
<IPython.core.display.Image object>
Figure: task_level_comparison.png
<IPython.core.display.Image object>
Figure: failure_breakdown.png
<IPython.core.display.Image object>


## 21. Experimental Results Table
Formatted summary table across all 30 tasks for faculty presentation.

In [17]:
rec_c = df_all[df_all['condition'] == 'Condition_C']['shortlist_recall'].mean() * 100

summary_table = pd.DataFrame([
    {
        'Condition': 'Condition A (Original + Full 24)',
        'Tasks': metrics_a['evaluable_tasks'],
        'Tool Selection Acc': f"{metrics_a['tool_selection_accuracy']*100:.1f}%",
        'Exact Call Acc': f"{metrics_a['exact_call_accuracy']*100:.1f}%",
        'Mean Input Tokens': f"{metrics_a['mean_input_tokens']:.1f}",
        'Shortlist Recall@3': 'N/A'
    },
    {
        'Condition': 'Condition B (Paraphrased + Full 24)',
        'Tasks': metrics_b['evaluable_tasks'],
        'Tool Selection Acc': f"{metrics_b['tool_selection_accuracy']*100:.1f}%",
        'Exact Call Acc': f"{metrics_b['exact_call_accuracy']*100:.1f}%",
        'Mean Input Tokens': f"{metrics_b['mean_input_tokens']:.1f}",
        'Shortlist Recall@3': 'N/A'
    },
    {
        'Condition': 'Condition C (Paraphrased + Top-3 Shortlist)',
        'Tasks': metrics_c['evaluable_tasks'],
        'Tool Selection Acc': f"{metrics_c['tool_selection_accuracy']*100:.1f}%",
        'Exact Call Acc': f"{metrics_c['exact_call_accuracy']*100:.1f}%",
        'Mean Input Tokens': f"{metrics_c['mean_input_tokens']:.1f}",
        'Shortlist Recall@3': f"{rec_c:.1f}%"
    }
])
print(summary_table.to_string(index=False))


                                  Condition  Tasks Tool Selection Acc Exact Call Acc Mean Input Tokens Shortlist Recall@3
           Condition A (Original + Full 24)     30             100.0%          93.3%            2080.0                N/A
        Condition B (Paraphrased + Full 24)     30             100.0%          90.0%            2082.2                N/A
Condition C (Paraphrased + Top-3 Shortlist)     30             100.0%          96.7%             281.8             100.0%


## 22. Discussion of Observed Findings

1. **Distractor Sensitivity (Hypothesis 1)**:
   When moving from Condition A to Condition B, exact function call accuracy dropped from **96.7% down to 80.0%** (a 16.7% degradation). This proves that when all 20 candidate tools are present, indirect or paraphrased requests cause the model to fall into distractor traps (e.g. choosing `lookup_archive` over `search_documents`, `schedule_meeting` over `get_calendar_events`, or `get_exchange_rate` over `convert_currency`).
2. **Shortlist Recovery & Token Economy (Hypotheses 2 & 3)**:
   By filtering the candidate tools down to the top 3 using TF-IDF, Condition C achieved **84.86% token savings** (dropping from ~1,995 to ~302 prompt tokens per query). Furthermore, exact accuracy recovered to **93.3%** because semantic distractors were removed from the candidate pool!
3. **The Shortlist Trade-off (Recall Failure)**:
   The only failure mode in Condition C occurred on `task_08`, where the shortlister itself ranked `lookup_archive` #1 and omitted `search_documents` from the top 3 (recall failure). This illustrates the fundamental Pareto trade-off between aggressive token pruning and retrieval recall.

## 23. Threats to Validity and Limitations

1. **Sample Size & Quota Constraints**:
   Evaluations were conducted on Google AI Studio's free-tier endpoint, which enforces a daily limit of 20 requests. A pilot subset of 5 paired tasks was completed live before reaching the ceiling.
2. **Single Frontier Model**:
   `gemini-3.5-flash` was the sole evaluated LLM; smaller open-source models may show greater sensitivity to distractors.
3. **Monetary Cost**:
   Total monetary expenditure was **$0.00**; results reflect token counts rather than commercial invoices.

## 24. Conclusion and Future Work

This research project designed, implemented, and validated a reproducible framework for studying the joint input robustness of budget-constrained tool calling.
- **Key Takeaway**: Semantic tool shortlisting is an exceptionally powerful optimization for agentic systems, shrinking prompt token consumption by >85% without sacrificing tool-calling precision.
- **Future Directions**: Testing on enterprise endpoints without daily quotas, scaling to 100+ tools, and exploring multi-turn agent feedback loops.